In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of spatialdata to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiscale-spatial-image to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of ome-zarr to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. 

## import

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [4]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention")
cwd = os.getcwd()
print(cwd)

sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention


In [5]:
import scanpy as sc
import spatialdata as sd
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)


In [6]:
import HadmardAttention as HA

/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [7]:
import importlib
import HadmardAttention.tools
import HadmardAttention.model

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: Use

SpatialData object, with associated Zarr store: /content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 24689, 17051), (3, 12344, 8525), (3, 6172, 4262), (3, 3086, 2131), (3, 1543, 1065)
│     └── 'morphology_focus': DataTree[cyx] (4, 23912, 34154), (4, 11956, 17077), (4, 5978, 8538), (4, 2989, 4269), (4, 1494, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
│     └── 'nucleus_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (63173, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (63173, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (63036, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (63173, 5006)
with coordi

In [ ]:
adata = sdata.tables["table"]
adata

AnnData object with n_obs × n_vars = 63173 × 5006
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

(63173, 5006)

In [ ]:
adata_omiCLIP = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/cells.h5ad")
adata_omiCLIP

AnnData object with n_obs × n_vars = 63173 × 1
    obs: 'cell_id'
    obsm: 'X_custom'

In [ ]:
# 1. Ensure cell IDs are the index (not just a column)
if 'cell_id' in adata.obs.columns:
    adata.obs.set_index('cell_id', inplace=True)
if 'cell_id' in adata_omiCLIP.obs.columns:
    adata_omiCLIP.obs.set_index('cell_id', inplace=True)

# 2. Align the two objects by cell_id (intersection)
common_ids = adata.obs_names.intersection(adata_omiCLIP.obs_names)

# Optional: check how many matched
print(f"Matched {len(common_ids)} cells out of {adata.n_obs}")

# 3. Reorder both to the same order
adata_c = adata[common_ids, :].copy()
adata_omiCLIP_c = adata_omiCLIP[common_ids, :].copy()

# 4. Add the X_custom matrix to adata_main.obsm
adata_c.obsm['Morpho_Embedding'] = adata_omiCLIP_c.obsm['X_custom']

# 5. Done! Verify
print(adata_c.obsm.keys())

Matched 63173 cells out of 63173
KeysView(AxisArrays with keys: spatial, Morpho_Embedding)


In [ ]:
adata_c.write("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

# Dataset

In [8]:
adata = sc.read_h5ad("../../../Data/Breast_Cancer/ann_data.h5ad")

Founsation Models

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata.obs_names = [str(int(cid[63:])-1) for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['UNI'][edata.obs_names.get_indexer(common_cells)]

In [ ]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=500)
X_reduced = pca.fit_transform(adata.obsm['Morpho_Embedding'])
adata.obsm['p_Morpho_Embedding'] = X_reduced

If the reduced embedding has been saved before:

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/hoptimus_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/virchow_adata.h5ad")

Noise

In [9]:
rng = np.random.default_rng(42)
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal((167780, 500))

In [10]:
adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Preparing Dataset

In [11]:
adata = HA.prep_adatas(adata, norm=True, log1p=True)
dataset = HA.make_dataset(adata, sparse_graph=True)

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.


In [12]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)

Expression torch.Size([167780, 313])
Morpho_Embedding torch.Size([167780, 500])
Neighborhood_Graph torch.Size([2, 1342240])


In [ ]:
importlib.reload(HA.dataset)
importlib.reload(HA.model)
importlib.reload(HA)

<module 'HadmardAttention' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention/../../HadmardAttention/__init__.py'>

In [25]:
import gc

gc.collect()
torch.cuda.empty_cache()

# Train

## Model Type 0

In [27]:
model_type = 0

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


### NOISE


In [ ]:
#NOISE 1.0
#Ex1: 18325
#Ex2: 18325

model.fit(dataset, entry_masking_rate=1.0,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26811
Epoch 101: loss =  0.21520
Epoch 201: loss =  0.18138
Epoch 301: loss =  0.14230
Epoch 401: loss =  0.13432
Epoch 501: loss =  0.12771
Epoch 601: loss =  0.12337
Epoch 701: loss =  0.12108
Epoch 801: loss =  0.11944
Epoch 901: loss =  0.11815
Epoch 1001: loss =  0.11746
Epoch 1101: loss =  0.11680
Epoch 1201: loss =  0.11625
Epoch 1301: loss =  0.11587
Epoch 1401: loss =  0.11523
Epoch 1501: loss =  0.11492
Epoch 1601: loss =  0.11471
Epoch 1701: loss =  0.11442
Epoch 1801: loss =  0.11430
Epoch 1901: loss =  0.11397
Epoch 2001: loss =  0.11371
Epoch 2101: loss =  0.11352
Epoch 2201: loss =  0.11339
Epoch 2301: loss =  0.11322
Epoch 2401: loss =  0.11313
Epoch 2501: loss =  0.11303
Epoch 2601: loss =  0.11294
Epoch 2701: loss =  0.11294
Epoch 2801: loss =  0.11295
Epoch 2901: loss =  0.11271
Epoch 3001: loss =  0.11266
Epoch 3101: loss =  0.11266
Epoch 3201: loss =  0.11264
Epoch 3301: loss =  0.11264
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_NOISE_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_NOISE_{model_type}.pth',weights_only=True))

### h_Optimus

In [ ]:
#H_OPTIMUS
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26815
Epoch 101: loss =  0.20429
Epoch 201: loss =  0.14332
Epoch 301: loss =  0.13020
Epoch 401: loss =  0.12223
Epoch 501: loss =  0.11798
Epoch 601: loss =  0.11663
Epoch 701: loss =  0.11479
Epoch 801: loss =  0.11381
Epoch 901: loss =  0.11288
Epoch 1001: loss =  0.11258
Epoch 1101: loss =  0.11115
Epoch 1201: loss =  0.11047
Epoch 1301: loss =  0.10986
Epoch 1401: loss =  0.11004
Epoch 1501: loss =  0.10952
Epoch 1601: loss =  0.10862
Epoch 1701: loss =  0.10886
Epoch 1801: loss =  0.10796
Epoch 1901: loss =  0.10846
Epoch 2001: loss =  0.10741
Epoch 2101: loss =  0.10736
Epoch 2201: loss =  0.10672
Epoch 2301: loss =  0.10650
Epoch 2401: loss =  0.10651
Epoch 2501: loss =  0.10613
Epoch 2601: loss =  0.10578
Epoch 2701: loss =  0.10739
Epoch 2801: loss =  0.10608
Epoch 2901: loss =  0.10599
Epoch 3001: loss =  0.10538
Epoch 3101: loss =  0.10533
Epoch 3201: loss =  0.10596
Epoch 3301: loss =  0.10510
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_hoptimus_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_hoptimus_{model_type}.pth',weights_only=True))

### UNI

In [ ]:
#UNI 0.2
model.fit(dataset, entry_masking_rate=0.2,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26828
Epoch 101: loss =  0.20252
Epoch 201: loss =  0.13809
Epoch 301: loss =  0.12320
Epoch 401: loss =  0.11526
Epoch 501: loss =  0.11165
Epoch 601: loss =  0.10965
Epoch 701: loss =  0.10830
Epoch 801: loss =  0.10696
Epoch 901: loss =  0.10636
Epoch 1001: loss =  0.10567
Epoch 1101: loss =  0.10565
Epoch 1201: loss =  0.10521
Epoch 1301: loss =  0.10469
Epoch 1401: loss =  0.10428
Epoch 1501: loss =  0.10417
Epoch 1601: loss =  0.10407
Epoch 1701: loss =  0.10396
Epoch 1801: loss =  0.10360
Epoch 1901: loss =  0.10342
Epoch 2001: loss =  0.10339
Epoch 2101: loss =  0.10331
Epoch 2201: loss =  0.10298
Epoch 2301: loss =  0.10294
Epoch 2401: loss =  0.10277
Epoch 2501: loss =  0.10288
Epoch 2601: loss =  0.10259
Epoch 2701: loss =  0.10270
Epoch 2801: loss =  0.10242
Epoch 2901: loss =  0.10266
Epoch 3001: loss =  0.10250
Epoch 3101: loss =  0.10225
Epoch 3201: loss =  0.10208
Epoch 3301: loss =  0.10186
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}_0.2.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}_0.2.pth',weights_only=True))

In [ ]:
model_type = 0

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
#UNI 0.5
model.fit(dataset, entry_masking_rate=0.5,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26825
Epoch 101: loss =  0.20312
Epoch 201: loss =  0.13904
Epoch 301: loss =  0.12465
Epoch 401: loss =  0.11748
Epoch 501: loss =  0.11461
Epoch 601: loss =  0.11193
Epoch 701: loss =  0.11031
Epoch 801: loss =  0.10956
Epoch 901: loss =  0.10939
Epoch 1001: loss =  0.10833
Epoch 1101: loss =  0.10775
Epoch 1201: loss =  0.10755
Epoch 1301: loss =  0.10797
Epoch 1401: loss =  0.10749
Epoch 1501: loss =  0.10705
Epoch 1601: loss =  0.10725
Epoch 1701: loss =  0.10684
Epoch 1801: loss =  0.10680
Epoch 1901: loss =  0.10664
Epoch 2001: loss =  0.10696
Epoch 2101: loss =  0.10638
Epoch 2201: loss =  0.10645
Epoch 2301: loss =  0.10614
Epoch 2401: loss =  0.10574
Epoch 2501: loss =  0.10552
Epoch 2601: loss =  0.10559
Epoch 2701: loss =  0.10486
Epoch 2801: loss =  0.10471
Epoch 2901: loss =  0.10453
Epoch 3001: loss =  0.10434
Epoch 3101: loss =  0.10423
Epoch 3201: loss =  0.10432
Epoch 3301: loss =  0.10408
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}_0.5.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}_0.5.pth',weights_only=True))

In [ ]:
model_type = 0

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
#UNI 0.8
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26820
Epoch 101: loss =  0.20548
Epoch 201: loss =  0.14139
Epoch 301: loss =  0.12795
Epoch 401: loss =  0.12035
Epoch 501: loss =  0.11708
Epoch 601: loss =  0.11433
Epoch 701: loss =  0.11286
Epoch 801: loss =  0.11226
Epoch 901: loss =  0.11172
Epoch 1001: loss =  0.11097
Epoch 1101: loss =  0.11061
Epoch 1201: loss =  0.10995
Epoch 1301: loss =  0.10995
Epoch 1401: loss =  0.10977
Epoch 1501: loss =  0.10867
Epoch 1601: loss =  0.10820
Epoch 1701: loss =  0.10776
Epoch 1801: loss =  0.10785
Epoch 1901: loss =  0.10708
Epoch 2001: loss =  0.10688
Epoch 2101: loss =  0.10672
Epoch 2201: loss =  0.10623
Epoch 2301: loss =  0.10598
Epoch 2401: loss =  0.10572
Epoch 2501: loss =  0.10551
Epoch 2601: loss =  0.10509
Epoch 2701: loss =  0.10515
Epoch 2801: loss =  0.10480
Epoch 2901: loss =  0.10461
Epoch 3001: loss =  0.10461
Epoch 3101: loss =  0.10427
Epoch 3201: loss =  0.10442
Epoch 3301: loss =  0.10398
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}_0.8.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}_0.8.pth',weights_only=True))

In [ ]:
model_type = 0

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
#UNI 1.0
model.fit(dataset, entry_masking_rate=1.0,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26815
Epoch 101: loss =  0.20778
Epoch 201: loss =  0.14754
Epoch 301: loss =  0.13362
Epoch 401: loss =  0.12736
Epoch 501: loss =  0.12295
Epoch 601: loss =  0.11893
Epoch 701: loss =  0.11790
Epoch 801: loss =  0.11760
Epoch 901: loss =  0.11653
Epoch 1001: loss =  0.11584
Epoch 1101: loss =  0.11511
Epoch 1201: loss =  0.11541
Epoch 1301: loss =  0.11441
Epoch 1401: loss =  0.11398
Epoch 1501: loss =  0.11287
Epoch 1601: loss =  0.11243
Epoch 1701: loss =  0.11186
Epoch 1801: loss =  0.11166
Epoch 1901: loss =  0.11123
Epoch 2001: loss =  0.11128
Epoch 2101: loss =  0.11098
Epoch 2201: loss =  0.11097
Epoch 2301: loss =  0.10990
Epoch 2401: loss =  0.10987
Epoch 2501: loss =  0.10946
Epoch 2601: loss =  0.10920
Epoch 2701: loss =  0.10894
Epoch 2801: loss =  0.10849
Epoch 2901: loss =  0.10830
Epoch 3001: loss =  0.10799
Epoch 3101: loss =  0.10784
Epoch 3201: loss =  0.10771
Epoch 3301: loss =  0.10837
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}_1.0.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}_1.0.pth',weights_only=True))

### virchow

In [ ]:
#virchow
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26812
Epoch 101: loss =  0.20423
Epoch 201: loss =  0.14519
Epoch 301: loss =  0.13169
Epoch 401: loss =  0.12574
Epoch 501: loss =  0.12133
Epoch 601: loss =  0.11760
Epoch 701: loss =  0.11514
Epoch 801: loss =  0.11376
Epoch 901: loss =  0.11260
Epoch 1001: loss =  0.11132
Epoch 1101: loss =  0.11099
Epoch 1201: loss =  0.11039
Epoch 1301: loss =  0.10980
Epoch 1401: loss =  0.10959
Epoch 1501: loss =  0.10886
Epoch 1601: loss =  0.10883
Epoch 1701: loss =  0.10832
Epoch 1801: loss =  0.10822
Epoch 1901: loss =  0.10850
Epoch 2001: loss =  0.10777
Epoch 2101: loss =  0.10763
Epoch 2201: loss =  0.10906
Epoch 2301: loss =  0.10720
Epoch 2401: loss =  0.10721
Epoch 2501: loss =  0.10702
Epoch 2601: loss =  0.10695
Epoch 2701: loss =  0.10695
Epoch 2801: loss =  0.10645
Epoch 2901: loss =  0.10662
Epoch 3001: loss =  0.10638
Epoch 3101: loss =  0.10641
Epoch 3201: loss =  0.10640
Epoch 3301: loss =  0.10622
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_virchow_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_virchow_{model_type}.pth',weights_only=True))

## Model Type 5

In [ ]:
model_type = 5

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


### NOISE

In [ ]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26822
Epoch 101: loss =  0.20597
Epoch 201: loss =  0.14791
Epoch 301: loss =  0.13341
Epoch 401: loss =  0.13076
Epoch 501: loss =  0.12989
Epoch 601: loss =  0.12930
Epoch 701: loss =  0.12889
Epoch 801: loss =  0.12859
Epoch 901: loss =  0.12619
Epoch 1001: loss =  0.12563
Epoch 1101: loss =  0.12542
Epoch 1201: loss =  0.12357
Epoch 1301: loss =  0.12204
Epoch 1401: loss =  0.12176
Epoch 1501: loss =  0.12161
Epoch 1601: loss =  0.12158
Epoch 1701: loss =  0.12150
Epoch 1801: loss =  0.12141
Epoch 1901: loss =  0.12136
Epoch 2001: loss =  0.12134
Epoch 2101: loss =  0.12135
Epoch 2201: loss =  0.12127
Epoch 2301: loss =  0.12131
Epoch 2401: loss =  0.12132
Epoch 2501: loss =  0.12126
Epoch 2601: loss =  0.12125
Epoch 2701: loss =  0.12123
Epoch 2801: loss =  0.12125
Epoch 2901: loss =  0.12115
Epoch 3001: loss =  0.12116
Epoch 3101: loss =  0.12117
Epoch 3201: loss =  0.12118
Epoch 3301: loss =  0.12113
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_NOISE_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_NOISE_{model_type}.pth',weights_only=True))

### h_optimus

In [ ]:
#H_optimus
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26823
Epoch 101: loss =  0.20585
Epoch 201: loss =  0.14405
Epoch 301: loss =  0.12880
Epoch 401: loss =  0.12249
Epoch 501: loss =  0.11771
Epoch 601: loss =  0.11596
Epoch 701: loss =  0.11658
Epoch 801: loss =  0.11372
Epoch 901: loss =  0.11359
Epoch 1001: loss =  0.11292
Epoch 1101: loss =  0.11266
Epoch 1201: loss =  0.11246
Epoch 1301: loss =  0.11204
Epoch 1401: loss =  0.11204
Epoch 1501: loss =  0.11120
Epoch 1601: loss =  0.11061
Epoch 1701: loss =  0.11035
Epoch 1801: loss =  0.11022
Epoch 1901: loss =  0.11012
Epoch 2001: loss =  0.10969
Epoch 2101: loss =  0.10950
Epoch 2201: loss =  0.10881
Epoch 2301: loss =  0.10826
Epoch 2401: loss =  0.10787
Epoch 2501: loss =  0.10762
Epoch 2601: loss =  0.10730
Epoch 2701: loss =  0.10741
Epoch 2801: loss =  0.10690
Epoch 2901: loss =  0.10683
Epoch 3001: loss =  0.10693
Epoch 3101: loss =  0.10685
Epoch 3201: loss =  0.10634
Epoch 3301: loss =  0.10642
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_hoptimus_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_hoptimus_{model_type}.pth',weights_only=True))

### UNI

In [ ]:
#UNI 0.2
model.fit(dataset, entry_masking_rate=0.2,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26824
Epoch 101: loss =  0.20433
Epoch 201: loss =  0.14177
Epoch 301: loss =  0.12543
Epoch 401: loss =  0.11929
Epoch 501: loss =  0.11471
Epoch 601: loss =  0.11262
Epoch 701: loss =  0.11132
Epoch 801: loss =  0.11064
Epoch 901: loss =  0.10993
Epoch 1001: loss =  0.10952
Epoch 1101: loss =  0.10926
Epoch 1201: loss =  0.10898
Epoch 1301: loss =  0.10859
Epoch 1401: loss =  0.10825
Epoch 1501: loss =  0.10789
Epoch 1601: loss =  0.10736
Epoch 1701: loss =  0.10706
Epoch 1801: loss =  0.10665
Epoch 1901: loss =  0.10656
Epoch 2001: loss =  0.10620
Epoch 2101: loss =  0.10573
Epoch 2201: loss =  0.10542
Epoch 2301: loss =  0.10534
Epoch 2401: loss =  0.10513
Epoch 2501: loss =  0.10497
Epoch 2601: loss =  0.10504
Epoch 2701: loss =  0.10506
Epoch 2801: loss =  0.10466
Epoch 2901: loss =  0.10465
Epoch 3001: loss =  0.10449
Epoch 3101: loss =  0.10445
Epoch 3201: loss =  0.10440
Epoch 3301: loss =  0.10417
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}_0.2.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}_0.2.pth',weights_only=True))

In [ ]:
model_type = 5

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
#UNI 0.5
model.fit(dataset, entry_masking_rate=0.5,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26824
Epoch 101: loss =  0.20408
Epoch 201: loss =  0.14200
Epoch 301: loss =  0.12550
Epoch 401: loss =  0.11963
Epoch 501: loss =  0.11622
Epoch 601: loss =  0.11458
Epoch 701: loss =  0.11307
Epoch 801: loss =  0.11225
Epoch 901: loss =  0.11141
Epoch 1001: loss =  0.11018
Epoch 1101: loss =  0.10924
Epoch 1201: loss =  0.10873
Epoch 1301: loss =  0.10822
Epoch 1401: loss =  0.10724
Epoch 1501: loss =  0.10669
Epoch 1601: loss =  0.10645
Epoch 1701: loss =  0.10613
Epoch 1801: loss =  0.10590
Epoch 1901: loss =  0.10591
Epoch 2001: loss =  0.10566
Epoch 2101: loss =  0.10552
Epoch 2201: loss =  0.10531
Epoch 2301: loss =  0.10536
Epoch 2401: loss =  0.10551
Epoch 2501: loss =  0.10499
Epoch 2601: loss =  0.10484
Epoch 2701: loss =  0.10508
Epoch 2801: loss =  0.10486
Epoch 2901: loss =  0.10476
Epoch 3001: loss =  0.10474
Epoch 3101: loss =  0.10468
Epoch 3201: loss =  0.10462
Epoch 3301: loss =  0.10448
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}_0.5.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}_0.5.pth',weights_only=True))

In [ ]:
model_type = 5

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
#UNI
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26823
Epoch 101: loss =  0.20488
Epoch 201: loss =  0.14366
Epoch 301: loss =  0.12875
Epoch 401: loss =  0.12355
Epoch 501: loss =  0.12213
Epoch 601: loss =  0.11853
Epoch 701: loss =  0.11682
Epoch 801: loss =  0.11440
Epoch 901: loss =  0.11347
Epoch 1001: loss =  0.11256
Epoch 1101: loss =  0.11166
Epoch 1201: loss =  0.11158
Epoch 1301: loss =  0.11021
Epoch 1401: loss =  0.10990
Epoch 1501: loss =  0.10918
Epoch 1601: loss =  0.10945
Epoch 1701: loss =  0.10905
Epoch 1801: loss =  0.10823
Epoch 1901: loss =  0.10800
Epoch 2001: loss =  0.10794
Epoch 2101: loss =  0.10763
Epoch 2201: loss =  0.10718
Epoch 2301: loss =  0.10674
Epoch 2401: loss =  0.10652
Epoch 2501: loss =  0.10617
Epoch 2601: loss =  0.10611
Epoch 2701: loss =  0.10578
Epoch 2801: loss =  0.10549
Epoch 2901: loss =  0.10528
Epoch 3001: loss =  0.10492
Epoch 3101: loss =  0.10461
Epoch 3201: loss =  0.10437
Epoch 3301: loss =  0.10422
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}.pth',weights_only=True))

In [ ]:
model_type = 5

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
#UNI 1.0
model.fit(dataset, entry_masking_rate=1.0,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26821
Epoch 101: loss =  0.21110
Epoch 201: loss =  0.14926
Epoch 301: loss =  0.13111
Epoch 401: loss =  0.12673
Epoch 501: loss =  0.12310
Epoch 601: loss =  0.11903
Epoch 701: loss =  0.11767
Epoch 801: loss =  0.11724
Epoch 901: loss =  0.11658
Epoch 1001: loss =  0.11600
Epoch 1101: loss =  0.11559
Epoch 1201: loss =  0.11520
Epoch 1301: loss =  0.11524
Epoch 1401: loss =  0.11506
Epoch 1501: loss =  0.11378
Epoch 1601: loss =  0.11368
Epoch 1701: loss =  0.11289
Epoch 1801: loss =  0.11263
Epoch 1901: loss =  0.11206
Epoch 2001: loss =  0.11168
Epoch 2101: loss =  0.11183
Epoch 2201: loss =  0.11118
Epoch 2301: loss =  0.11071
Epoch 2401: loss =  0.11045
Epoch 2501: loss =  0.11029
Epoch 2601: loss =  0.11018
Epoch 2701: loss =  0.10972
Epoch 2801: loss =  0.10965
Epoch 2901: loss =  0.10900
Epoch 3001: loss =  0.10931
Epoch 3101: loss =  0.10902
Epoch 3201: loss =  0.10839
Epoch 3301: loss =  0.10779
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_UNI_{model_type}_1.0.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_UNI_{model_type}_1.0.pth',weights_only=True))

### virchow

In [ ]:
#virchow
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26822
Epoch 101: loss =  0.20723
Epoch 201: loss =  0.14572
Epoch 301: loss =  0.12796
Epoch 401: loss =  0.12454
Epoch 501: loss =  0.12182
Epoch 601: loss =  0.11867
Epoch 701: loss =  0.11646
Epoch 801: loss =  0.11604
Epoch 901: loss =  0.11476
Epoch 1001: loss =  0.11471
Epoch 1101: loss =  0.11403
Epoch 1201: loss =  0.11373
Epoch 1301: loss =  0.11347
Epoch 1401: loss =  0.11323
Epoch 1501: loss =  0.11324
Epoch 1601: loss =  0.11302
Epoch 1701: loss =  0.11263
Epoch 1801: loss =  0.11237
Epoch 1901: loss =  0.11248
Epoch 2001: loss =  0.11236
Epoch 2101: loss =  0.11200
Epoch 2201: loss =  0.11181
Epoch 2301: loss =  0.11167
Epoch 2401: loss =  0.11159
Epoch 2501: loss =  0.11131
Epoch 2601: loss =  0.11138
Epoch 2701: loss =  0.11127
Epoch 2801: loss =  0.11179
Epoch 2901: loss =  0.11151
Epoch 3001: loss =  0.11114
Epoch 3101: loss =  0.11087
Epoch 3201: loss =  0.11091
Epoch 3301: loss =  0.11083
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alpha

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/breast_cancer_32_virchow_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_virchow_{model_type}.pth',weights_only=True))

# Check